##### Copyright 2026 Google LLC.

In [2]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.


# File Search Quickstart

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/File_Search.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>


The [File Search tool](http://ai.google.dev/gemini-api/docs/file-search) allows you to build powerful retrieval-augmented generation (RAG) applications using Gemini. It lets you upload documents to a managed store and then use them as a tool during model generation, enabling Gemini to answer questions based on your specific data with accurate citations.

In this quickstart, you will learn how to:

*   Create a File Search Store.
*   Upload documents to the store.
*   Use the store as a tool in `interactions.create`.
*   Cite the sources used during generation.
*   Filter search results using custom metadata.
*   Manage your documents and stores.

For information on how pricing works for File Search, including details on what is available free of charge, see the [pricing information](https://ai.google.dev/gemini-api/docs/file-search#pricing).

> **Note:** This notebook uses the [Interactions API](https://ai.google.dev/gemini-api/docs/interactions), the latest way to interact with Gemini models. Looking for the `generateContent` version? Check the [archive branch](https://github.com/google-gemini/cookbook/blob/archive/generate-content-api/quickstarts/File_Search.ipynb).

## Install dependencies

First, install the Google Gen AI SDK.


In [7]:
# SDK 1.75.0 has latest multimodal File Search features
%pip install -U -q "google-genai>=2.9.0"  # 1.75+ for Interactions API + File Search

### Authentication

**Important:** The File Search API uses API keys for authentication and access. Uploaded files are associated with the API key's cloud project. Unlike other Gemini APIs that use API keys, your API key also grants access to all data you've uploaded to file stores, so take extra care in keeping your API key secure. For best practices on securing API keys, refer to Google's [documentation](https://support.google.com/googleapi/answer/6310037).

#### Set up your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for a walkthrough.

In [10]:
from google import genai
from google.colab import userdata
from google.genai import types

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(api_key=GEMINI_API_KEY)

## Basic file search

In this section, you will download a sample document, create a File Search Store, and use it to answer questions.

### Create a File Search Store

Create a new File Search Store to hold your documents.


In [13]:
file_search_store = client.file_search_stores.create(
    config=types.CreateFileSearchStoreConfig(
        display_name='My File Search Store'
    )
)

print(f"Created store: {file_search_store.name}")

Created store: fileSearchStores/my-file-search-store-zl14xbhbfu41


### Download a sample document

Download "A Survey of Modernist Poetry" from Project Gutenberg as a sample text file.

In [15]:
!wget -q https://www.gutenberg.org/cache/epub/76401/pg76401.txt -O sample_poetry.txt
!head sample_poetry.txt

### Upload a file to the store

Upload the text file directly to the store. The ingestion process includes some processing, so you need to wait for it to complete before you can search.

In [17]:
import time

upload_op = client.file_search_stores.upload_to_file_search_store(
    file_search_store_name=file_search_store.name,
    file='sample_poetry.txt',
    config=types.UploadToFileSearchStoreConfig(
        display_name='A Survey of Modernist Poetry',
    )
)

print(f"Upload started: {upload_op.name}")


while not (upload_op := client.operations.get(upload_op)).done:
    time.sleep(1)
    print(".", end="")

print()
print("Processing complete.")

Upload started: fileSearchStores/my-file-search-store-zl14xbhbfu41/upload/operations/a-survey-of-modernist-poetr-pahgkrfkxior
...................................................................................................................................................................
Processing complete.


### Alternative: Import from the File API

If you have already uploaded documents to the [File API](https://ai.google.dev/gemini-api/docs/files), you can import them directly into a File Store. This may be helpful if a user has performed some interaction with a file already, such as generating a summary, and has approved the file for use in a store.

In [19]:
file_ref = client.files.upload(
    file='sample_poetry.txt',
    config=types.UploadFileConfig(
        display_name='A Survey of Modernist Poetry',
        mime_type='text/plain',
    )
)
print(f"Uploaded via File API: {file_ref.name}")

import_op = client.file_search_stores.import_file(
    file_search_store_name=file_search_store.name,
    file_name=file_ref.name,
)

print(f"File import started: {import_op.name}")

while not (import_op := client.operations.get(import_op)).done:
    time.sleep(1)
    print(".", end="")

print()
print("Processing complete.")

Uploaded via File API: files/d9jsb1hxh364
File import started: fileSearchStores/my-file-search-store-zl14xbhbfu41/operations/d9jsb1hxh364-340ginvi1as3
......
Processing complete.


### Generate content with File Search

Now, use the `file_search` tool in an `interactions.create` request to ask a question that requires information from the uploaded document.

In [21]:
MODEL_ID = "gemini-3.6-flash" # @param ["gemini-3.5-flash-lite", "gemini-2.5-flash", "gemini-3.6-flash", "gemini-2.5-flash-preview", "gemini-3-flash-preview", "gemini-2.5-pro", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

interaction = client.interactions.create(
    model=MODEL_ID,
    input='What does the text say about E.E. Cummings?',
    tools=[{"type": "file_search", "file_search_store_names": [file_search_store.name]}],
)

print(interaction.steps[-1].content[0].text)

The text discusses E.E. Cummings as a significant and challenging modernist poet whose work requires readers to adopt a new critical attitude. According to the documents, particularly *A Survey of Modernist Poetry*, the following points are made about him:

### **1. Relationship with the Reader**
*   **Demanding Style:** Cummings is described as a poet who uses "exceptional means" to force readers to move beyond "lazy reading habits" and common intelligence. His poetry demands a "vigorous imaginative effort" that the average reader may be unwilling to apply.
*   **Lack of Obligation:** Unlike some earlier modernists who aimed for simplicity for the sake of the "plain reader," Cummings represents a modernism that feels no obligation to the average reader, focusing instead on the interests of poetry itself.
*   **Critical Hostility:** His work often excites hostility. Even critics like Louis Untermeyer, who included Cummings in his *Anthology of Modern American Poetry*, are described as 

#### Additional fields

The `FileSearch` tool provides some options for configuring how the tool works, `top_k` and `metadata_filter`.

`top_k` controls how many chunks will be returned from the search tool and passed to the generation step. This is the same example as before, but only 1 chunk will be used to generate the answer. This control can be helpful to guide the model if you know there is only one correct chunk to consider (`k=1`), or if you expect the chunks to have more overlap and you want to include more context (higher `top_k`).

Metadata filtering is described in the next section, and you can find the full spec in the [API reference](https://ai.google.dev/api/caching#FileSearch).

In [23]:
top_K = 1 # @param {"allow-input":true, isTemplate: true}

interaction = client.interactions.create(
    model=MODEL_ID,
    input='What does the text say about E.E. Cummings?',
    tools=[{"type": "file_search", "file_search_store_names": [file_search_store.name], "top_k": top_K}],
)

print(interaction.steps[-1].content[0].text)

Based on the text, E.E. Cummings is presented as a modern poet whose work requires a significant shift in critical attitude and a "vigorous imaginative effort" from the reader. The text characterizes his poetry and its reception in several ways:

### **Critical Perspective and Innovations**
*   **A "New Kind of Poem":** Cummings is described as a poet who uses "exceptional means" and has essentially invented a "new kind of poem" to force readers to do justice to his work, rather than indulging "lazy reading habits" fostered by simple anthologies.
*   **Influence on Reading:** Even if not accepted for his own sake, the text argues that he must be accepted for his impact on how poetry—regardless of age or style—will be read in the future.
*   **Modernism:** His work represents a form of modernism that feels no obligation to the "plain reader." This contrasts with the modernism of earlier movements like Imagism, which sought simplicity and everyday language. Cummings’ modernism is instead

### Inspect grounding metadata and page numbers

The response includes `grounding_metadata` which contains citations and references to the source document. The document chunks used in the generation context are available in `grounding_metadata.grounding_chunks`, and look like this.

Each chunk also includes a `page_number` field, which tells you the exact page in the source document the chunk came from. This is especially useful for fact-checking and verification over large documents.

```python
[
  GroundingChunk(
    retrieved_context=GroundingChunkRetrievedContext(
      text="""(the snippet of text contained in this chunk)""",
      title='(the title of the document)',
      page_number=5
    )
  ), ...
]
```

In [25]:
import textwrap

# Note: Grounding metadata with detailed chunk information is not yet
# available in the Interactions API. Use generate_content API for grounding
print("File Search grounding metadata not yet available in Interactions API")
print("The response text is available via interaction.steps[-1].content[0].text")

File Search grounding metadata not yet available in Interactions API
The response text is available via interaction.steps[-1].content[0].text


Additionally, `grounding_metadata` includes `grounding_supports` that provide references from the response text to the supporting documents, and can be used for providing annotations.

The supports look like this.

```python
[
  GroundingSupport(
    grounding_chunk_indices=[
      0,  # The index in `grounding_chunks` to which this corresponds
    ],
    segment=Segment(
      start_index=123,  # Indices into the generated text
      end_index=456,
      text='(the span of generated text being supported)'
    )
  ), ...
]
```

In [27]:
from IPython.display import Markdown, display

# Annotated response display requires grounding metadata.
print("Annotated response display requires grounding metadata (not yet in Interactions API)")

Annotated response display requires grounding metadata (not yet in Interactions API)


## Metadata Filtering

While adding documents, you can attach custom metadata to your files and use it to filter search results.

### Upload a file with metadata

Download another book, "Alice's Adventures in Wonderland", and upload it with information about genre and author.

In [29]:
!wget -q https://www.gutenberg.org/files/11/11-0.txt -O alice_in_wonderland.txt
!head alice_in_wonderland.txt

In [30]:
upload_op = client.file_search_stores.upload_to_file_search_store(
    file_search_store_name=file_search_store.name,
    file='alice_in_wonderland.txt',
    config=types.UploadToFileSearchStoreConfig(
        display_name='Alice in Wonderland',
        custom_metadata=[
            types.CustomMetadata(key='genre', string_value='fiction'),
            types.CustomMetadata(key='author', string_value='Lewis Carroll'),
        ]
    )
)

while not (upload_op := client.operations.get(upload_op)).done:
    time.sleep(1)
    print(".", end="")

print()
print("Upload complete.")

..
Upload complete.


Custom metadata can be provided as `string_value`, `numeric_value` or `string_list_value` types.

In [32]:
types.CustomMetadata.model_fields.keys() - {'key'}

### Query with metadata filter

Now, ask a question that could apply to either book, but use a filter to restrict it to just one. For example, ask about a "Queen" but filter for 'fiction'.


In [34]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input='Who is the Queen?',
    tools=[{"type": "file_search", "file_search_store_names": [file_search_store.name], "metadata_filter": 'genre = "fiction"'}],
)

print(interaction.steps[-1].content[0].text)

The term "the Queen" can refer to several different figures depending on the context:

### 1. **Queen Camilla (United Kingdom and Commonwealth)**
As of 2026, the most prominent figure with this title is **Queen Camilla**. She is the wife of King Charles III. 
*   **Status:** She became **Queen Consort** upon accession of King Charles III on September 8, 2022, following the death of Queen Elizabeth II.
*   **Title:** Since the Coronation on May 6, 2023, she has been officially styled as **Queen Camilla**.

### 2. **Queen Elizabeth II (Historical context)**
For many, "the Queen" still refers to **Queen Elizabeth II**, who reigned for 70 years from 1952 until 2022. She was a **Queen Regnant**, meaning she reigned in her own right with full sovereign powers. Her long reign made her the most recognized "Queen" globally for decades.

### 3. **Other Current Queens (Consorts)**
Currently, there are no **Queens Regnant** (female monarchs who rule in their own right) in the world. However, there

To learn more about how to build complex filters, read the [AIP-160](https://google.aip.dev/160) spec.

## Multimodal File Search

With multimodal File Search, you can natively embed and search through both documents and images. By using the `gemini-embedding-2` embedding model, images are embedded directly (not via OCR), enabling true visual retrieval.

In this section, you'll create a multimodal File Search Store, upload images, query them with Gemini, and retrieve media citations.

### Create a multimodal File Search Store

To enable multimodal search, specify `gemini-embedding-2` as the `embedding_model` when creating a store. If omitted, the store defaults to `gemini-embedding-001`, which is optimized for text-only workloads.

| Embedding Model | Best For |
|---|---|
| `gemini-embedding-001` (default) | Text-heavy workloads, cost-optimized |
| `gemini-embedding-2` | Multimodal retrieval (text *and* images) |

In [38]:
mm_file_search_store = client.file_search_stores.create(
    config=types.CreateFileSearchStoreConfig(
        display_name='Multimodal Store',
        embedding_model='models/gemini-embedding-2'
    )
)

print(f"Created multimodal store: {mm_file_search_store.name}")

Created multimodal store: fileSearchStores/multimodal-store-qxos6g9bw4dx


### Upload images

Upload files to the store using `upload_to_file_search_store`. With `gemini-embedding-2`, this works for both documents and images (PNG, JPEG). Images within PDFs are also natively embedded alongside the text.

In [40]:
from IPython.display import Image, display

# Download a sample image
!curl -so sample_image.jpg "https://storage.googleapis.com/generativeai-downloads/images/jetpack.jpg"

display(Image("sample_image.jpg", width=300))

SyntaxError: unterminated string literal (detected at line 4) (File_Search.ipynb:cell_39, line 4)

In [41]:
# Upload the image to the multimodal store
upload_op = client.file_search_stores.upload_to_file_search_store(
    file_search_store_name=mm_file_search_store.name,
    file='sample_image.jpg',
    config=types.UploadToFileSearchStoreConfig(
        display_name='Jetpack Image',
    )
)

print(f"Upload started: {upload_op.name}")

while not (upload_op := client.operations.get(upload_op)).done:
    time.sleep(1)
    print(".", end="")

print()
print("Processing complete.")

FileNotFoundError: sample_image.jpg is not a valid file path.

### Query with multimodal File Search

Use the `file_search` tool to query the multimodal store. The model performs a semantic search over both text and image data to generate a grounded response.

In [43]:
interaction = client.interactions.create(
    model=MODEL_ID,
    input='Describe what is in the image stored in the file search store.',
    tools=[{"type": "file_search", "file_search_store_names": [mm_file_search_store.name]}],
)

print(interaction.steps[-1].content[0].text)

BadRequestError: Error code: 400 - {'error': {'message': 'Model generated too many tool calls. Please retry the request. If the issue persists, include this error message in the retry prompt to allow the model to call

### Retrieve media citations and page numbers

When the model references an image chunk, the grounding metadata includes a `media_id` field. You can use this to download the exact image the model used for its response.

For text citations, the metadata also includes a `page_number` field, which tells you the exact page in the source document the chunk came from. This is especially useful for fact-checking and verification over large documents.

In [45]:
# Grounding metadata for file search is not yet available in the Interactions API.
print("Grounding metadata not yet available in Interactions API")

Grounding metadata not yet available in Interactions API


## Managing Documents

You can also manage individual documents within a store.

### List documents

List all documents currently in the store.


In [47]:
print(f"Documents in {file_search_store.name}:")

for doc in client.file_search_stores.documents.list(parent=file_search_store.name):
    print(f"- {doc.display_name} ({doc.name})")

Documents in fileSearchStores/my-file-search-store-zl14xbhbfu41:
- A Survey of Modernist Poetry (fileSearchStores/my-file-search-store-zl14xbhbfu41/documents/a-survey-of-modernist-poetr-pahgkrfkxior)
- d9jsb1hxh364 (fileSearchStores/my-file-search-store-zl14xbhbfu41/documents/d9jsb1hxh364-340ginvi1as3)
- Alice in Wonderland (fileSearchStores/my-file-search-store-zl14xbhbfu41/documents/alice-in-wonderland-tz6xws9klm4r)


### Get a document

Retrieve details for a specific document, such as its processing status or metadata.


In [49]:
# Get a document by ID
doc_id = doc.name  # Or set a specific ID here.
sample_doc = client.file_search_stores.documents.get(name=doc_id)

if sample_doc:
    print(f"Document details for {sample_doc.display_name}:")
    print(f"  Name: {sample_doc.name}")
    print(f"  Custom Metadata: {sample_doc.custom_metadata}")

Document details for Alice in Wonderland:
  Name: fileSearchStores/my-file-search-store-zl14xbhbfu41/documents/alice-in-wonderland-tz6xws9klm4r
  Custom Metadata: [CustomMetadata(
  key='genre',
  string_value='fiction'
), CustomMetadata(
  key='author',
  string_value='Lewis Carroll'
)]


### Delete a document

Remove a specific document from the store without deleting the entire store.


In [51]:
# Delete a specific document.
doc_to_be_deleted = doc_id

client.file_search_stores.documents.delete(
    name=doc_to_be_deleted,
    config=types.DeleteDocumentConfig(
        # Set force to delete a non-empty document.
        force=True
    )
)
print(f"Deleted document: {doc_to_be_deleted}")

# Verify deletion
print("\nRemaining documents:")
for doc in client.file_search_stores.documents.list(parent=file_search_store.name):
    print(f"- {doc.display_name}")

Deleted document: fileSearchStores/my-file-search-store-zl14xbhbfu41/documents/alice-in-wonderland-tz6xws9klm4r

Remaining documents:
- A Survey of Modernist Poetry
- d9jsb1hxh364


## Managing File Search Stores

You can list, get, and delete your File Search Stores.


In [53]:
print("Stores:")

for store in client.file_search_stores.list():
    print(f"- {store.name} ({store.display_name})")

Stores:
- fileSearchStores/linkedinlivedemo-kz1t09dt27ty (linkedin_live_demo)
- fileSearchStores/myexamplestore-t0kby8lqatxp (my_example_store)
- fileSearchStores/linkedinlivedemostore-0hk474x2klnn (linkedin_live_demo_store)
- fileSearchStores/myexamplestore-ewfrht3ddbr9 (my_example_store)
- fileSearchStores/linkedinlivedemostore-p5zvredysq71 (linkedin_live_demo_store)
- fileSearchStores/myexamplestore-mvryze3utwrl (my_example_store)
- fileSearchStores/linkedinlivedemostore-76txfj2jemza (linkedin_live_demo_store)
- fileSearchStores/yourfilesearchstorename-kkntkxscsa22 (your-fileSearchStore-name)
- fileSearchStores/yourfilesearchstorename-hr6xgqu0ge6v (your-fileSearchStore-name)
- fileSearchStores/yourfilesearchstorename-u61bp32tqnpj (your-fileSearchStore-name)
- fileSearchStores/yourfilesearchstorename-2miaspfrlcwi (your-fileSearchStore-name)
- fileSearchStores/yourfilesearchstorename-jlma4n6bygmz (your-fileSearchStore-name)
- fileSearchStores/yourfilesearchstorename-rantys2z0h2s (your

## Cleanup

It's good practice to delete File Search Stores when you are done with them to avoid unnecessary storage costs.


In [55]:
client.file_search_stores.delete(name=file_search_store.name, config=types.DeleteFileSearchStoreConfig(force=True))
print(f"Deleted store: {file_search_store.name}")

client.file_search_stores.delete(name=mm_file_search_store.name, config=types.DeleteFileSearchStoreConfig(force=True))
print(f"Deleted store: {mm_file_search_store.name}")

Deleted store: fileSearchStores/my-file-search-store-zl14xbhbfu41
Deleted store: fileSearchStores/multimodal-store-qxos6g9bw4dx


## What's next

For more information on the File Search tool, be sure to check these out.
 * The [File Search guide](https://ai.google.dev/gemini-api/docs/file-search)
 * [Pricing information](https://ai.google.dev/gemini-api/docs/file-search#pricing)
 * The detailed [API reference](https://ai.google.dev/api/file-search)
 * The [File API cookbook](./File_API.ipynb)